# Tools, Retrieval, and Agents

A language model on its own can only produce text. This lab builds the three things that turn text into a *system*: **tools** (so the model can act), **retrieval** (so it can look things up), and **a loop** (so it can decide what to do next). By the end you will have a working ReAct agent with three tools, a step budget, and loop detection — and you will have watched it recover from three different failures.

Every model call here is a `FakeLLM`: a small deterministic rule-based stand-in with the same contract a real chat API has, **text in, text out**. There is no network in this kernel, and that is a feature — the traces are identical on every machine, and every line of plumbing you write around the stand-in is the same plumbing you would keep with a hosted model.

**How to use this notebook:** run cells top to bottom; later sections reuse earlier functions. Companion reading: Chapters 28, 29 and 30.

## 1. JSON Schema: the contract between model and program

A tool is a Python function, but the model cannot see Python. What it sees is a **JSON Schema**: a machine-readable description of the arguments, with types, required fields, enums and ranges. The schema does two jobs at once — it tells the model what to send, and it lets your program reject what should never reach the function.

Here is a validator for the subset of JSON Schema that tool definitions actually use. Writing it yourself is worth ten pages of documentation, because you find out exactly how little the specification is doing for you.

In [ ]:
def validate(value, schema, path="$"):
    """Return a list of human-readable problems. Empty list means valid."""
    problems = []
    kind = schema.get("type")

    if kind == "object":
        if not isinstance(value, dict):
            return [f"{path}: expected object, got {type(value).__name__}"]
        props = schema.get("properties", {})
        for key in schema.get("required", []):
            if key not in value:
                problems.append(f"{path}.{key}: required but missing")
        for key, sub in value.items():
            if key in props:
                problems += validate(sub, props[key], f"{path}.{key}")
            elif schema.get("additionalProperties") is False:
                problems.append(f"{path}.{key}: not allowed by the schema")
        return problems

    if kind == "array":
        if not isinstance(value, list):
            return [f"{path}: expected array, got {type(value).__name__}"]
        for i, item in enumerate(value):
            problems += validate(item, schema.get("items", {}), f"{path}[{i}]")
        return problems

    expected = {"string": str, "integer": int, "number": (int, float),
                "boolean": bool}.get(kind)
    if kind == "integer" and isinstance(value, bool):
        return [f"{path}: expected integer, got boolean"]   # bool is an int in Python!
    if expected is not None and not isinstance(value, expected):
        return [f"{path}: expected {kind}, got {type(value).__name__}"]

    if "enum" in schema and value not in schema["enum"]:
        problems.append(f"{path}: {value!r} not one of {schema['enum']}")
    if "minimum" in schema and value < schema["minimum"]:
        problems.append(f"{path}: {value} below minimum {schema['minimum']}")
    if "maximum" in schema and value > schema["maximum"]:
        problems.append(f"{path}: {value} above maximum {schema['maximum']}")
    return problems

ORDER_SCHEMA = {
    "type": "object",
    "properties": {
        "sku": {"type": "string", "enum": ["widget", "gizmo", "sprocket"]},
        "units": {"type": "integer", "minimum": 1, "maximum": 500},
        "express": {"type": "boolean"},
    },
    "required": ["sku", "units"],
    "additionalProperties": False,
}

CASES = [
    {"sku": "widget", "units": 12},                          # fine
    {"sku": "widget", "units": 12, "express": True},         # fine
    {"units": 12},                                           # missing required
    {"sku": "flange", "units": 12},                          # not in the enum
    {"sku": "gizmo", "units": 0},                            # below minimum
    {"sku": "gizmo", "units": "twelve"},                     # wrong type
    {"sku": "gizmo", "units": True},                         # sneaky: bool is an int
    {"sku": "gizmo", "units": 3, "rush": "yes"},             # unknown property
]
for case in CASES:
    errs = validate(case, ORDER_SCHEMA)
    print(f"{str(case):<50} {'VALID' if not errs else errs[0]}")

Two valid calls and six ways to be wrong — and each of the six is a real bug a model produces: it omits a field, invents an enum value, sends a number as a word, or adds a helpful extra key you never asked for. Two lines are worth pausing on.

`{"sku": "gizmo", "units": True}` is caught only because we checked for it. In Python `bool` is a *subclass* of `int`, so `isinstance(True, int)` is `True` and a naive validator would happily pass `True` as a unit count.

`{"rush": "yes"}` is caught only because the schema says `"additionalProperties": false`. Leave that out and unknown keys sail straight through into your `**kwargs` and raise a `TypeError` deep inside the tool, where the error message is useless to the model.

### Micro-exercise: a schema for a date range

Write `RANGE_SCHEMA` describing an object with required string fields `start` and `end` (dates as text), an optional integer `limit` between 1 and 100, and no other properties. Then check it against the three cases below.

In [ ]:
RANGE_SCHEMA = {
    "type": "object",
    # your code here: properties, required, additionalProperties
}

# Uncomment to test once the schema is filled in:
# for case in [{"start": "2024-01-01", "end": "2024-02-01"},
#              {"start": "2024-01-01"},
#              {"start": "2024-01-01", "end": "2024-02-01", "limit": 500}]:
#     print(case, "->", validate(case, RANGE_SCHEMA) or "VALID")

## 2. A tool dispatcher and the round trip

Now the loop. The model does **not** run the tool — it *asks* for a call, your program runs the real function and hands the result back, and the model continues with that result in its context. Three exchanges: model asks, program answers, model concludes.

The dispatcher is where the schema earns its keep, and it is also your **security boundary**. Validate first, then call; never `eval` a string the model produced.

In [ ]:
DIRECTORY = {"ana": "ana@example.com", "ben": "ben@example.com"}

def calculate(a, b, op):
    """Arithmetic on two numbers. Note: we branch on `op`, we never eval()."""
    if op == "divide" and b == 0:
        raise ValueError("division by zero")
    return {"add": a + b, "subtract": a - b,
            "multiply": a * b, "divide": a / b if b else None}[op]

def directory_lookup(name):
    if name.lower() not in DIRECTORY:
        raise KeyError(f"no entry for {name!r}; known: {sorted(DIRECTORY)}")
    return DIRECTORY[name.lower()]

TOOLS = {
    "calculate": {
        "fn": calculate,
        "description": "Do one arithmetic operation on two numbers.",
        "schema": {"type": "object",
                   "properties": {"a": {"type": "number"}, "b": {"type": "number"},
                                  "op": {"type": "string",
                                         "enum": ["add", "subtract",
                                                  "multiply", "divide"]}},
                   "required": ["a", "b", "op"], "additionalProperties": False},
    },
    "directory_lookup": {
        "fn": directory_lookup,
        "description": "Find a colleague's email address by first name.",
        "schema": {"type": "object",
                   "properties": {"name": {"type": "string"}},
                   "required": ["name"], "additionalProperties": False},
    },
}

def dispatch(name, arguments):
    """Validate, then run. Every failure comes back as text the model can read."""
    if name not in TOOLS:
        return (f"TOOL ERROR: no tool named {name!r}. "
                f"Available: {', '.join(sorted(TOOLS))}.")
    errs = validate(arguments, TOOLS[name]["schema"])
    if errs:
        return "SCHEMA ERROR: " + "; ".join(errs)
    try:
        return str(TOOLS[name]["fn"](**arguments))
    except Exception as exc:
        return f"TOOL ERROR: {type(exc).__name__}: {exc}"

for name, args in [("calculate", {"a": 18, "b": 4, "op": "multiply"}),
                   ("calculate", {"a": 18, "b": 0, "op": "divide"}),
                   ("calculate", {"a": 18, "b": 4, "op": "root"}),
                   ("directory_lookup", {"name": "Ana"}),
                   ("directory_lookup", {"name": "Zoe"}),
                   ("web_search", {"q": "anything"})]:
    print(f"{name}({args}) -> {dispatch(name, args)}")

Notice that **nothing raises**. Every failure — unknown tool, bad enum value, division by zero, missing person — comes back as a string, because that string is going into the model's context. An error message is the most useful prompt you will ever write: `"no tool named 'web_search'. Available: calculate, directory_lookup"` teaches the model how to recover, while a bare `KeyError` kills the process.

Now the stand-in model, and the round trip.

In [ ]:
import json

class FakeLLM:
    """A rule-based policy standing in for a chat model: text in, text out.

    Each rule fires on what is MISSING from the transcript, so the policy
    reacts to whatever the tools actually returned rather than replaying a
    fixed script. Swapping in a hosted model replaces one line of run_tools.
    """

    def __call__(self, transcript):
        if "@" not in transcript:
            return 'CALL directory_lookup {"name": "Ana"}'
        if "72" not in transcript:
            return 'CALL calculate {"a": 18, "b": 4, "op": "multiply"}'
        return ("ANSWER Ana's address is ana@example.com, and 18 x 4 = 72.")

def run_tools(llm, goal, max_turns=5):
    transcript = f"USER {goal}\n"
    for turn in range(1, max_turns + 1):
        reply = llm(transcript)                       # <- the one line to swap
        print(f"--- turn {turn} ---")
        print("model:", reply)
        if reply.startswith("ANSWER"):
            return reply[len("ANSWER"):].strip()
        _, name, blob = reply.split(" ", 2)
        observation = dispatch(name, json.loads(blob))
        print("program:", observation)
        transcript += f"{reply}\nRESULT {observation}\n"
    print("--- turn budget exhausted ---")
    return None

answer = run_tools(FakeLLM(), "What is Ana's email, and what is 18 times 4?")
print("\nfinal:", answer)

Three model calls, two tool runs, one answer — and the *order* was never written in the source. The policy looked at the transcript, noticed no email address was in it yet, and asked for one. Change `DIRECTORY` so the lookup fails and the same policy will see the error text instead and behave differently, with no code change.

## 3. MCP: the same idea, over a wire

Tools that live in the same file as the loop are the easy case. Real tools live in a database, a ticket tracker, a filesystem — and if every AI application writes its own integration for every system you get $M \times N$ pieces of bespoke plumbing. The **Model Context Protocol** (MCP, an open standard introduced by Anthropic in late 2024) puts a protocol in the middle so the cost becomes $M + N$: each application learns the protocol once, each system gets wrapped in a **server** once.

MCP does not invent a message format; it uses **JSON-RPC 2.0**, which has exactly three shapes. A **request** has `method`, optional `params`, and an `id`. A **response** carries the same `id` back plus *either* `result` or `error`, never both. A **notification** is a request with **no `id`**, so no reply is expected or allowed.

The server below implements three methods for real. It is "mini" only in that messages arrive as Python dicts instead of newline-delimited JSON on a pipe — every dict is a spec-shaped JSON-RPC message.

In [ ]:
PROTOCOL_VERSION = "2025-06-18"          # MCP revisions are date-stamped

# JSON-RPC 2.0 reserved error codes.
PARSE_ERROR, INVALID_REQUEST = -32700, -32600
METHOD_NOT_FOUND, INVALID_PARAMS, INTERNAL_ERROR = -32601, -32602, -32603

class MiniMCPServer:
    """An MCP-style server: initialize, tools/list, tools/call."""

    def __init__(self, name, version="0.1.0"):
        self.name, self.version = name, version
        self.tools = {}

    def add_tool(self, name, description, input_schema, fn):
        self.tools[name] = {"description": description,
                            "inputSchema": input_schema, "fn": fn}

    def handle(self, message):
        """One message in, one response out - or None for a notification."""
        if message.get("jsonrpc") != "2.0":
            return self._error(message.get("id"), INVALID_REQUEST,
                               "jsonrpc must be exactly '2.0'")
        method, mid = message.get("method"), message.get("id")
        params = message.get("params", {})
        if mid is None:                       # a notification: act, stay silent
            return None
        try:
            if method == "initialize":
                return self._ok(mid, {
                    "protocolVersion": PROTOCOL_VERSION,
                    "capabilities": {"tools": {"listChanged": False}},
                    "serverInfo": {"name": self.name, "version": self.version}})
            if method == "tools/list":
                return self._ok(mid, {"tools": [
                    {"name": n, "description": t["description"],
                     "inputSchema": t["inputSchema"]}
                    for n, t in sorted(self.tools.items())]})
            if method == "tools/call":
                return self._call_tool(mid, params)
        except Exception as exc:              # a server must never die
            return self._error(mid, INTERNAL_ERROR, f"{type(exc).__name__}: {exc}")
        return self._error(mid, METHOD_NOT_FOUND, f"unknown method: {method!r}",
                           {"available": ["initialize", "tools/list", "tools/call"]})

    def _call_tool(self, mid, params):
        name = params.get("name")
        if name not in self.tools:
            return self._error(mid, INVALID_PARAMS, f"no tool named {name!r}",
                               {"available": sorted(self.tools)})
        args = params.get("arguments", {})
        missing = [k for k in self.tools[name]["inputSchema"].get("required", [])
                   if k not in args]
        if missing:
            return self._error(mid, INVALID_PARAMS, "missing required arguments",
                               {"missing": missing})
        try:
            value = self.tools[name]["fn"](**args)
        except Exception as exc:
            # A tool that FAILS is a result, not a protocol error: the model is
            # meant to read the message and try something else.
            return self._ok(mid, {"isError": True, "content": [
                {"type": "text", "text": f"{type(exc).__name__}: {exc}"}]})
        return self._ok(mid, {"isError": False, "content": [
            {"type": "text", "text": str(value)}]})

    @staticmethod
    def _ok(mid, result):
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    @staticmethod
    def _error(mid, code, message, data=None):
        err = {"code": code, "message": message}
        if data is not None:
            err["data"] = data
        return {"jsonrpc": "2.0", "id": mid, "error": err}

INVENTORY = {"widget": 42, "gizmo": 7, "sprocket": 0}
PRICES = {"widget": 3.50, "gizmo": 19.99, "sprocket": 0.75}

def stock_level(sku):
    if sku not in INVENTORY:
        raise KeyError(f"unknown sku {sku!r}")
    return INVENTORY[sku]

def restock_cost(sku, units):
    return round(PRICES[sku] * units, 2)

server = MiniMCPServer("inventory-server", "1.0.0")
server.add_tool("stock_level",
                "Return how many units of one SKU are in the warehouse now.",
                {"type": "object",
                 "properties": {"sku": {"type": "string",
                                        "enum": sorted(INVENTORY)}},
                 "required": ["sku"]},
                stock_level)
server.add_tool("restock_cost",
                "Compute the USD cost of ordering N units of one SKU.",
                {"type": "object",
                 "properties": {"sku": {"type": "string", "enum": sorted(INVENTORY)},
                                "units": {"type": "integer", "minimum": 1}},
                 "required": ["sku", "units"]},
                restock_cost)
print("server", repr(server.name), "exposes:", ", ".join(sorted(server.tools)))

Now the conversation, message by message. Watch the `id` field: it is what matches a response to the request that caused it, because both sides may have several messages in flight and replies can arrive in any order.

In [ ]:
import json

CONVERSATION = [
    # 1. The handshake. Nothing else may be sent until this completes.
    {"jsonrpc": "2.0", "id": 1, "method": "initialize",
     "params": {"protocolVersion": PROTOCOL_VERSION,
                "capabilities": {"roots": {"listChanged": True}},
                "clientInfo": {"name": "handbook-host", "version": "0.1.0"}}},
    # 2. A notification: no id, so no reply is expected or allowed.
    {"jsonrpc": "2.0", "method": "notifications/initialized"},
    # 3. Discovery: what can this server do?
    {"jsonrpc": "2.0", "id": 2, "method": "tools/list"},
    # 4. A successful call.
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call",
     "params": {"name": "stock_level", "arguments": {"sku": "widget"}}},
    # 5. Missing argument -> a PROTOCOL error (-32602), before the tool runs.
    {"jsonrpc": "2.0", "id": 4, "method": "tools/call",
     "params": {"name": "restock_cost", "arguments": {"sku": "gizmo"}}},
    # 6. The tool itself raises -> a RESULT with isError, not a protocol error.
    {"jsonrpc": "2.0", "id": 5, "method": "tools/call",
     "params": {"name": "stock_level", "arguments": {"sku": "flange"}}},
    # 7. A method the server never claimed to have.
    {"jsonrpc": "2.0", "id": 6, "method": "resources/read",
     "params": {"uri": "file:///etc/passwd"}},
]

for message in CONVERSATION:
    print("-> " + json.dumps(message))
    response = server.handle(message)
    if response is None:
        print("<- (no reply: notifications have no id)")
    else:
        print("<- " + json.dumps(response))
    print()

Four things in that transcript are the whole protocol.

**The handshake is capability negotiation.** The client announces which date-stamped revision it speaks and what it can do; the server replies with the version it will actually use and its own capabilities. That is why a client written against an old revision can talk to a new server: both sides stay inside the intersection, and nobody calls `resources/read` on a server that never claimed to have resources — which is exactly what message 7 gets told, with $-32601$ and a list of the methods that do exist.

**The notification gets no reply at all.** No `id`, no response. Fire and forget.

**Missing arguments and a failing tool are different failures.** Message 5 never reaches `restock_cost`: the server checks `required` first and returns error $-32602$ with a `data` field naming exactly what was missing. Message 6 *does* reach `stock_level`, which raises — and that comes back as a successful `result` with `isError: true`. The distinction matters because a protocol error means "the client is broken", while `isError` means "the model should read this and try something else".

### Micro-exercise: a third tool

Add a `reorder_point(sku, days)` tool to `server` that returns `days * 3` as a suggested reorder quantity, with an integer `days` of minimum 1, then call it through `handle` — once correctly, once with `days` missing.

In [ ]:
def reorder_point(sku, days):
    # your code here: return a suggested quantity, e.g. days * 3
    return 0

# your code here: server.add_tool("reorder_point", ..., reorder_point)

# Uncomment to test once the tool is registered:
# print(server.handle({"jsonrpc": "2.0", "id": 7, "method": "tools/call",
#                      "params": {"name": "reorder_point",
#                                 "arguments": {"sku": "gizmo", "days": 5}}}))
# print(server.handle({"jsonrpc": "2.0", "id": 8, "method": "tools/call",
#                      "params": {"name": "reorder_point",
#                                 "arguments": {"sku": "gizmo"}}}))

## 4. Retrieval: searching by meaning, not by string

Keyword search matches strings, and strings are not meanings: a user who types "automobile" gets nothing from a document that says "car". The fix is to turn every document into a **vector** and rank by the angle between vectors.

We cannot run an embedding model in this kernel, so we build the vectorizer that predates them and still backs half of production search: **TF-IDF**. One dimension per vocabulary word, the value is how often the word occurs, weighted by how *rare* the word is across the corpus:

$$
\text{tf-idf}(w, d) = \underbrace{\text{count of } w \text{ in } d}_{\text{term frequency}} \times \underbrace{\left(\ln\frac{1+N}{1+\text{df}(w)} + 1\right)}_{\text{inverse document frequency}}
$$

A word in every document earns a weight near 1; a word in one document earns a large one. Rare words carry the signal. Normalize every vector to length 1 once, at insert time, and cosine similarity becomes a plain dot product.

In [ ]:
import re
import numpy as np

CORPUS = [
    "Electric cars store energy in a large lithium battery pack.",
    "Battery range drops noticeably in very cold winter weather.",
    "A petrol engine burns fuel and wastes most of it as heat.",
    "Charging a battery at home overnight is cheaper than a fast charger.",
    "Sourdough bread needs a starter culture of wild yeast.",
    "Bread dough rises because yeast produces carbon dioxide gas.",
    "Cold weather slows down yeast, so dough rises much more slowly.",
    "A hot oven sets the crust of the bread in the first ten minutes.",
    "Python lists store references to objects, not the objects themselves.",
    "A dictionary looks up a key in constant time using a hash.",
    "Sorting a list of one million items takes a fraction of a second.",
    "Reading a large file line by line keeps memory usage small.",
]

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

vocab = sorted({w for doc in CORPUS for w in tokenize(doc)})
column = {w: i for i, w in enumerate(vocab)}
N = len(CORPUS)

doc_freq = np.zeros(len(vocab))
for doc in CORPUS:
    for w in set(tokenize(doc)):          # set(): count each document once
        doc_freq[column[w]] += 1
idf = np.log((1 + N) / (1 + doc_freq)) + 1.0

def vectorize(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in column:
            v[column[w]] += 1.0           # term frequency
    v *= idf                              # weight by rarity
    length = np.linalg.norm(v)
    return v / length if length > 0 else v    # normalize -> cosine == dot product

INDEX = np.stack([vectorize(d) for d in CORPUS])
print("index shape (documents x vocabulary):", INDEX.shape)
print("idf of 'a':", round(float(idf[column["a"]]), 3),
      "  idf of 'lithium':", round(float(idf[column["lithium"]]), 3))

def search(query, k=3):
    q = vectorize(query)
    scores = INDEX @ q                    # one dot product per document
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]

for query in ["cold weather battery", "why does dough rise", "automobile"]:
    print(f"\nquery: {query!r}")
    for rank, (i, s) in enumerate(search(query), start=1):
        print(f"  {rank}. score={s:.4f}  {CORPUS[i]}")

The first query works: the cold-battery sentence comes first, the cold-yeast sentence second. Then read the other two, because they show both of TF-IDF's limits.

`'why does dough rise'` ranks the dough sentences first — but only because of the word *dough*. The query word `rise` never matches the document word `rises`; TF-IDF compares exact strings, which is why classic search engines add **stemming**.

`'automobile'` scores **0.0000** on every document. The word is not in the vocabulary, so its vector is all zeros, so every dot product is zero; the ranking you see is `argsort` breaking a twelve-way tie. This failure is *structural* to lexical vectors, and it is precisely what a trained embedding model fixes: it has no vocabulary axis, so it maps "automobile" near "car" because the training text used them alike. Everything else here — normalization, dot products, top-$k$ — stays exactly the same when you swap the vectorizer. That is why the machinery can be taught offline.

## 5. Chunking: where RAG quietly breaks

Retrieval does not fetch documents, it fetches **chunks**. Choosing where to cut is the least glamorous and most consequential decision in a RAG pipeline, because a fact split across a boundary can no longer be retrieved by either half.

Below, one specific sentence answers one specific question. We chunk the document three ways and ask, for each, whether *any single chunk* still contains the whole answer.

In [ ]:
DOC = ("The Atlas ingestion service accepts documents in batches. "
       "Throughput is stable under load. "
       "The maximum batch size is 64 documents per request. "
       "Larger batches are rejected with HTTP 413. "
       "Retries use exponential backoff.")
ANSWER = "maximum batch size is 64 documents"

def fixed_size(text, size, overlap=0):
    """Cut every `size` characters, optionally repeating `overlap` characters."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]

def by_sentence(text, max_chars):
    """Pack whole sentences into chunks, never splitting one."""
    out, current = [], ""
    for s in text.split(". "):
        s = s if s.endswith(".") else s + "."
        if current and len(current) + 1 + len(s) > max_chars:
            out.append(current)
            current = s
        else:
            current = (current + " " + s).strip()
    if current:
        out.append(current)
    return out

def report(label, chunks):
    survives = any(ANSWER in c for c in chunks)
    print(f"{label:<34} {len(chunks):>2} chunks   answer retrievable: {survives}")
    for i, c in enumerate(chunks):
        print(f"    [{i}] {c!r}")
    print()

print(f"the answer we must be able to retrieve: {ANSWER!r}\n")
report("fixed 110 chars, no overlap", fixed_size(DOC, 110))
report("fixed 110 chars, 30 overlap", fixed_size(DOC, 110, overlap=30))
report("sentence-aware, 110 chars", by_sentence(DOC, 110))

The first strategy destroys the answer, and `answer retrievable: False` says so. Look at where the cut lands: chunk 0 ends `...The maximum batch s` and chunk 1 begins `ize is 64 documents...`. Neither chunk contains the fact, so no query about batch size can retrieve it. Worse, chunk 0 is *confidently retrievable and useless* — it looks like it is about batch size and contains no number, so the model will either say it does not know or invent one.

Overlap is the cheapest insurance in RAG: repeat the last few hundred characters of each chunk at the start of the next, and a fact can only be lost if it is longer than the overlap. Thirty characters is enough here, and it costs storage plus a little duplicate retrieval, nothing else.

Sentence-aware chunking is better still, because it respects a boundary the *author* put there. The general rule: split on the largest structural unit that fits — sections, then paragraphs, then sentences — and never on a fixed character count if you can help it.

### Micro-exercise: find the smallest safe overlap

For overlaps from 0 to 60 in steps of 5, check whether `fixed_size(DOC, 90, overlap)` keeps `ANSWER` inside some single chunk, and print the smallest overlap that works. Compare it with `len(ANSWER)`.

In [ ]:
def smallest_safe_overlap(size=90):
    for overlap in range(0, 65, 5):
        # your code here: chunk with this overlap, return it if ANSWER survives
        pass
    return None

# Uncomment to test:
# print("smallest safe overlap:", smallest_safe_overlap(), " len(ANSWER) =", len(ANSWER))

## 6. Memory: a context window is a budget

An agent's memory is a transcript, and you resend all of it on every model call. So the prompt grows linearly in turns while the *total* tokens sent grow with the square of the turn count — and eventually the transcript simply does not fit.

The simplest fix is a **sliding window**: keep the last few turns, drop the rest. It is one line, it never fails, and it forgets things that matter.

In [ ]:
CONVERSATION = [
    "User: hi, I'm Rosa and I'm on the Atlas team.\n",
    "Assistant: hello Rosa - what can I help with?\n",
    "User: my timezone is UTC+1, keep that in mind for schedules.\n",
    "Assistant: noted, UTC+1.\n",
    "User: the ingestion service is rejecting batches with HTTP 413.\n",
    "Assistant: that means the batch exceeded the maximum of 64 documents.\n",
    "User: I use zsh, so give me zsh-compatible commands.\n",
    "Assistant: understood, zsh it is.\n",
    "User: can you check whether a restart is safe right now?\n",
    "Assistant: a restart drains in-flight batches first, so it is safe.\n",
    "User: how long does the drain take?\n",
    "Assistant: usually under two minutes.\n",
]
FACTS = {"name": "Rosa", "timezone": "UTC+1", "shell": "zsh", "limit": "64"}

def n_tokens(text):
    """Rough English rule of thumb: about 4 characters per token."""
    return len(text) // 4

def facts_kept(text):
    return sorted(k for k, v in FACTS.items() if v in text)

def sliding_window(turns, budget):
    kept = []
    for turn in reversed(turns):              # newest first
        if n_tokens("".join(kept)) + n_tokens(turn) > budget:
            break
        kept.insert(0, turn)
    return kept

print(f"whole conversation: {n_tokens(''.join(CONVERSATION))} tokens, "
      f"{len(CONVERSATION)} turns")
print(f"{'budget':>7}{'turns kept':>12}{'tokens':>8}   facts still in context")
for budget in [200, 100, 60, 30]:
    kept = sliding_window(CONVERSATION, budget)
    print(f"{budget:>7}{len(kept):>12}{n_tokens(''.join(kept)):>8}   "
          f"{facts_kept(''.join(kept))}")

Read the last column. As the budget tightens, the window drops the *oldest* turns first — and the oldest turns are exactly where a user states durable facts about themselves. At a tight budget the agent no longer knows Rosa's name, timezone, or shell, and will cheerfully hand her a bash command in the wrong timezone. The information was never wrong; it was evicted.

Summarization keeps the old turns' *content* while paying for only a fraction of their tokens. A real implementation calls the model to write the summary; ours extracts the durable facts with rules, which is a `FakeSummarizer` — deterministic, offline, and honest about being a stand-in.

In [ ]:
import re

class FakeSummarizer:
    """Stands in for 'call the model and ask it to summarise these turns'.

    A real summariser is a model call. This one pattern-matches the durable
    facts, which is enough to show the budget arithmetic that matters.
    """

    RULES = [(r"I'm (\w+)", "user is {}"),
             (r"timezone is ([\w+\-]+)", "timezone {}"),
             (r"I use (\w+)", "shell {}"),
             (r"maximum of (\d+) documents", "batch limit {}")]

    def __call__(self, turns):
        text = "".join(turns)
        found = []
        for pattern, template in self.RULES:
            m = re.search(pattern, text)
            if m:
                found.append(template.format(m.group(1)))
        return "Summary of earlier turns: " + "; ".join(found) + ".\n"

def budgeted_memory(turns, budget, keep_verbatim=4, summarizer=FakeSummarizer()):
    """Keep the last `keep_verbatim` turns word for word; summarise the rest."""
    recent, older = turns[-keep_verbatim:], turns[:-keep_verbatim]
    parts = ([summarizer(older)] if older else []) + recent
    while n_tokens("".join(parts)) > budget and len(parts) > 1:
        parts.pop(1)                          # drop oldest verbatim turn, keep summary
    return parts

print(f"{'budget':>7}{'tokens':>8}{'parts':>7}   facts still in context")
for budget in [200, 100, 60, 30]:
    parts = budgeted_memory(CONVERSATION, budget)
    print(f"{budget:>7}{n_tokens(''.join(parts)):>8}{len(parts):>7}   "
          f"{facts_kept(''.join(parts))}")

print("\nwhat the model actually sees at budget 60:")
print("".join(budgeted_memory(CONVERSATION, 60)))

Same budgets, same conversation, and the facts survive all the way down. The summary is a handful of tokens carrying what a hundred tokens of transcript carried; the last few turns stay verbatim because recent wording is where the *current* task lives.

Two practical notes. Summarize **older** turns and keep recent ones verbatim, never the other way round. And keep the stable part of your prompt — system instructions, tool definitions, the summary — at the *front* and unchanged between calls, because a server's prefix cache can only reuse an exact prefix: one timestamp near the top destroys every cache hit after it.

## 7. The agent loop: ReAct

An **agent** is a program in which a model repeatedly chooses an action, observes the result, and uses the observation to choose the next action, until it decides it is done or a budget runs out. The novelty is not the loop; it is that the **control flow is data produced by the model at run time**.

The dominant text format comes from **ReAct** (Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023): interleave reasoning and acting so that each thought is informed by a real observation and each action is justified by a thought. The model writes `Thought`, `Action` and `Action Input`. **Your program writes the `Observation` lines** — it parses the action, runs the real function, and appends the result. The transcript is the memory.

Three real tools first.

In [ ]:
WIKI = {
    "eiffel tower": "The Eiffel Tower stands in Paris, France. Height: 330 m.",
    "empire state building": "The Empire State Building is in New York. Height: 443 m.",
    "great pyramid": "The Great Pyramid of Giza stands near Cairo. Height: 139 m.",
}

def wiki(query):
    """Look a landmark up in a three-article in-memory encyclopedia."""
    key = query.strip().strip('"').lower()
    for title, article in WIKI.items():
        if key in title or title in key:
            return article
    return f"No article titled {query!r}. Known titles: {', '.join(WIKI)}."

def calc(expression):
    """Arithmetic on two numbers. We parse the three parts rather than eval()."""
    m = re.fullmatch(r"\s*(-?[\d.]+)\s*([-+*/])\s*(-?[\d.]+)\s*", expression)
    if not m:
        return "Error: calc takes '<number> <op> <number>', e.g. '12 * 4'."
    a, op, b = float(m[1]), m[2], float(m[3])
    if op == "/" and b == 0:
        return "Error: division by zero."
    return str(round({"+": a + b, "-": a - b, "*": a * b,
                      "/": a / b if b else 0.0}[op], 4))

FACTORS = {("m", "ft"): 3.28084, ("ft", "m"): 0.3048,
           ("km", "mi"): 0.621371, ("kg", "lb"): 2.20462}

def convert(text):
    """Convert a quantity between units, e.g. '330 m to ft'."""
    m = re.fullmatch(r"\s*([\d.]+)\s*(\w+)\s+to\s+(\w+)\s*", text)
    if not m:
        return "Error: expected '<number> <unit> to <unit>', e.g. '5 m to ft'."
    amount, src, dst = float(m[1]), m[2], m[3]
    if (src, dst) not in FACTORS:
        return f"Error: I cannot convert {src} to {dst}. Known: {list(FACTORS)}."
    return f"{amount} {src} = {round(amount * FACTORS[(src, dst)], 2)} {dst}"

REACT_TOOLS = {"wiki": wiki, "calc": calc, "convert": convert}
for name, arg in [("wiki", "Eiffel Tower"), ("convert", "330 m to ft"),
                  ("calc", "1082.68 / 100"), ("wiki", "Eifel Tower")]:
    print(f"{name}({arg!r}) -> {REACT_TOOLS[name](arg)}")

Every tool returns a *string*, including its errors, and the error strings name the alternatives. That is not politeness — those strings go into the model's context, so they are prompts.

Now the parser and the loop. The parser has three outcomes: a final answer, an action, or `"unparsed"` — because a model that misformats its reply must be told so rather than crashing the run.

In [ ]:
def parse(reply):
    """Pull Thought / Action / Action Input / Final Answer out of a reply."""
    fields = {}
    for line in reply.splitlines():
        for tag in ("Thought", "Action Input", "Action", "Final Answer"):
            if line.strip().startswith(tag + ":"):
                fields[tag] = line.strip()[len(tag) + 1:].strip()
                break
    if "Final Answer" in fields:
        return {"kind": "final", **fields}
    if "Action" in fields and "Action Input" in fields:
        return {"kind": "act", **fields}
    return {"kind": "unparsed", **fields}          # the repair path

class ReactLLM:
    """Rule-based policy: each rule fires on what the transcript still lacks."""

    def __call__(self, transcript):
        if "Height:" not in transcript:
            return ("Thought: I need the tower's height before I can convert it.\n"
                    "Action: wiki\n"
                    "Action Input: Eiffel Tower")
        if " ft" not in transcript:
            return ("Thought: The article says 330 m. Convert that to feet.\n"
                    "Action: convert\n"
                    "Action Input: 330 m to ft")
        if "10.8268" not in transcript:
            return ("Thought: The height is 1082.68 ft; now divide it by 100.\n"
                    "Action: calc\n"
                    "Action Input: 1082.68 / 100")
        return ("Thought: I have the height in feet and the division.\n"
                "Final Answer: The Eiffel Tower is about 1082.68 ft tall, "
                "which is 10.8268 hundreds of feet.")

def run_react(llm, goal, tools, max_steps=6):
    transcript, seen = f"Goal: {goal}\n", set()
    for step in range(1, max_steps + 1):
        reply = llm(transcript)                    # <- the one line to swap
        decision = parse(reply)
        print(f"--- step {step} ---")
        print("Thought:", decision.get("Thought", "(none given)"))
        if decision["kind"] == "final":
            print("Final Answer:", decision["Final Answer"])
            return decision["Final Answer"]
        if decision["kind"] == "unparsed":
            observation = ("PARSE ERROR: reply with an 'Action:' line and an "
                           "'Action Input:' line, or with 'Final Answer:'.")
        else:
            action, arg = decision["Action"], decision["Action Input"]
            print(f"Action: {action}({arg!r})")
            fingerprint = (action, arg.strip().lower())        # the loop guard
            if fingerprint in seen:
                observation = (f"LOOP GUARD: you already ran {action}({arg!r}) "
                               "and it did not help. Try something different.")
            elif action not in tools:
                observation = (f"TOOL ERROR: no tool named {action!r}. "
                               f"Available tools: {', '.join(tools)}.")
            else:
                seen.add(fingerprint)
                observation = tools[action](arg)
        print("Observation:", observation)
        transcript += reply + f"\nObservation: {observation}\n"
    print(f"--- budget exhausted after {max_steps} steps, no answer ---")
    return None

goal = ("How tall is the Eiffel Tower in feet, and what is that height "
        "divided by 100?")
run_react(ReactLLM(), goal, REACT_TOOLS)

Three tools, three actions, one answer — and nowhere in the source did we write "first look it up, then convert, then divide". That ordering was *chosen*, one step at a time, from what came back. Change the encyclopedia entry to give the height in feet already and the policy skips a step with no code change.

Swapping in a hosted model touches exactly one line, `reply = llm(transcript)`. Everything else — the parser, the budget, the loop guard, the tool validation — is provider-independent plumbing you would keep as written.

Now break it on purpose, twice, because these are the two failures you will actually meet.

In [ ]:
class StubbornLLM:
    """A realistic bug: it never decides it is done."""
    def __call__(self, transcript):
        return ("Thought: Let me just double-check the height.\n"
                "Action: wiki\n"
                "Action Input: Eiffel Tower")

class TypoLLM:
    """Misspells the title, repeats the mistake, then reacts to the guard."""
    def __call__(self, transcript):
        if "Height:" in transcript:
            return "Thought: Found it.\nFinal Answer: 330 m."
        if "LOOP GUARD" not in transcript:
            return ("Thought: Look the tower up.\n"
                    "Action: wiki\nAction Input: Eifel Tower")        # typo
        return ("Thought: That exact call already failed; fix my spelling.\n"
                "Action: wiki\nAction Input: Eiffel Tower")

print("=== a policy with no stopping condition, budget 3 ===")
run_react(StubbornLLM(), "How tall is the Eiffel Tower?", REACT_TOOLS, max_steps=3)

print("\n=== a policy that repeats a failed action ===")
run_react(TypoLLM(), "How tall is the Eiffel Tower?", REACT_TOOLS, max_steps=5)

The first run ends with `budget exhausted`, and the way it gets there is the lesson. The loop guard fires on step 2 and again on step 3 — but `StubbornLLM` ignores the transcript entirely, so the guard changes nothing. **A guard only helps a policy that reads it; the budget helps regardless.** That is why `max_steps` is in the signature from the very first draft: it is the difference between a slow answer and a process you have to kill. Real budgets are usually a pair — max steps **and** max tokens or dollars.

The second run is more interesting. Step 1 misspells the title and gets a helpful "No article titled…" back. Step 2 repeats the *identical* call — and the guard, not the encyclopedia, answers, because the fingerprint `(tool, normalised input)` has been seen. Step 3 reads `LOOP GUARD` in the transcript and changes its input; step 4 finishes. The guard turned an infinite rut into a two-step detour, and it did so by **writing into the transcript** — the only channel you have to the model's next decision.

Note the cost of normalising the fingerprint with `.strip().lower()`: two genuinely distinct calls that differ only in case or spacing are treated as a repeat. For a tool where whitespace is meaningful — a shell command, a password check — fingerprint the raw input instead.

## What you built

| Piece | The idea in one sentence |
|---|---|
| JSON Schema validator | The schema is both documentation for the model and a gate for your program. |
| Dispatcher | Validate, then call; return every failure as text the model can read. |
| MCP / JSON-RPC | One protocol in the middle turns $M \times N$ integrations into $M + N$. |
| TF-IDF search | Normalize once, then similarity is a dot product; out-of-vocabulary words score zero. |
| Chunking | A fact split across a boundary cannot be retrieved by either half. |
| Budgeted memory | Summarize old turns, keep recent ones verbatim, keep the prefix stable. |
| ReAct loop | The model writes Thought/Action; the program writes Observation. |
| Budget + loop guard | A step budget stops runaway loops; a fingerprint stops repeated ones. |

## Try it yourself

Bigger exercises. Every scaffold runs as-is.

### Exercise 1 — A tool the agent has to discover

Add a `distance(a, b)` tool to `REACT_TOOLS` that returns the straight-line distance between two of three hard-coded cities, then write a policy that answers "How far is Paris from Cairo, in miles?" It will need `distance` and then `convert`. Make each rule fire on what the transcript *lacks*, not on a step counter — then check the trace still works if you reorder the rules.

In [ ]:
CITY_KM = {("paris", "cairo"): 3210, ("paris", "new york"): 5837,
           ("cairo", "new york"): 9020}

def distance(text):
    """'Paris to Cairo' -> a distance in km."""
    parts = [p.strip().lower() for p in text.lower().split(" to ")]
    # your code here: look the pair up in either order, return a string,
    # and return a helpful error naming the known pairs if it is missing
    return "Error: distance() has no lookup yet - fill it in above."

# REACT_TOOLS["distance"] = distance

class DistanceLLM:
    def __call__(self, transcript):
        # your code here: ask for the distance, then convert km to mi, then answer
        return ("Thought: this policy has no rules yet.\n"
                "Final Answer: fill in the rules above and re-run.")

# Uncomment to test:
# run_react(DistanceLLM(), "How far is Paris from Cairo, in miles?", REACT_TOOLS)

### Exercise 2 — Measure the quadratic cost of a transcript

The prompt at step $k$ holds roughly $k$ steps' worth of text, and you send a prompt on every step, so the total sent is $1 + 2 + \dots + n = n(n+1)/2$ steps' worth — quadratic in $n$ while the transcript itself is only linear. Instrument `run_react` (copy it, do not edit the original) to accumulate `n_tokens(transcript)` before each model call, print a table of prompt size and cumulative tokens per step, and extrapolate to 30 steps.

In [ ]:
def run_react_metered(llm, goal, tools, max_steps=6):
    transcript, sent = f"Goal: {goal}\n", 0
    print(f"{'step':>4} | {'chars':>6} | {'prompt tokens':>13} | {'sent so far':>12}")
    for step in range(1, max_steps + 1):
        # your code here: add n_tokens(transcript) to sent, print the row,
        # then do one llm/parse/tool round and grow the transcript
        break
    return sent

# Uncomment to test:
# total = run_react_metered(ReactLLM(), goal, REACT_TOOLS)
# print("total tokens sent:", total)

### Exercise 3 — Retrieve, then answer, with citations you can check

Wire retrieval into the tool loop: give the agent a `lookup(query)` tool that returns the top 2 TF-IDF hits from `CORPUS` **prefixed with their chunk numbers**, then have the policy answer using only those chunks and cite them as `[3]`. Finally write `verify_citations(answer, n_chunks)` that returns the cited numbers that do not exist — the cheapest hallucination check in RAG, and one you should always run.

In [ ]:
def lookup(query):
    hits = search(query, k=2)
    # your code here: return a string like "[0] first sentence\n[4] second sentence"
    return ""

def verify_citations(answer, n_chunks):
    cited = {int(m) for m in re.findall(r"\[(\d+)\]", answer)}
    # your code here: return the sorted cited numbers that are out of range
    return []

# Uncomment to test:
# print(lookup("cold weather battery"))
# print(verify_citations("As [1] says, batteries lose range. See also [99].", 12))

### Exercise 4 — Two retrievers are better than one

Lexical and semantic search fail in *different* directions: exact rare tokens (part numbers, function names) favour lexical, paraphrase favours semantic. **Reciprocal rank fusion** merges two ranked lists using ranks only, never scores, because scores from different retrievers are on incomparable scales:

$$
\text{RRF}(d) = \sum_{r} \frac{1}{k + \text{rank}_r(d)}, \qquad k \approx 60
$$

Implement `rrf(rankings, k=60)`, then fuse the TF-IDF ranking with a crude "topic" ranking that scores documents by how many words they share with a small hand-written lexicon. Check that the fused order beats both on the query `'cold battery'`.

In [ ]:
TOPIC_LEXICON = {
    "vehicle": "cars automobile vehicle engine petrol fuel battery charging "
               "charger lithium range pack".split(),
    "baking": "bread dough yeast sourdough oven crust starter".split(),
    "weather": "cold winter weather hot freezing temperature".split(),
}

def rrf(rankings, k=60):
    """Combine ranked lists of document ids using ranks only."""
    fused = {}
    # your code here: for each ranking, for each (rank, doc), add 1/(k+rank)
    return sorted(fused, key=lambda d: -fused[d]), fused

# Uncomment once rrf is written:
# lexical = [i for i, s in search("cold battery", k=12) if s > 0]
# print("lexical ranking:", lexical)
# print("fused:", rrf([lexical, [1, 0, 3, 6]])[0][:3])